# Notebook v3 — menjawab empat permintaan editor JUTI

Notebook ini **melanjutkan** v2, bukan menggantikannya. Bagian 0–3 (unduh, bersihkan, split) identik,
sehingga checkpoint yang sudah ada di folder `juti_rerun` dipakai ulang dan tidak dilatih ulang.

Empat permintaan editor dan bagian yang menjawabnya:

| Permintaan editor | Bagian | Keluaran |
|---|---|---|
| 1. Baseline yang lebih kuat (XGBoost / backbone neural) | **G** | semua tabel terkontrol dilaporkan untuk 3 keluarga model |
| 2. Tumpang tindih fitur pada Infiltration | **I** | Fisher ratio, N1, N2, N3 per kategori + eksperimen representasi |
| 3. Analisis tujuan kesalahan yang sistematis | **H** | tabel tujuan false negative untuk semua kategori dan semua model |
| 4. Penjelasan korelasi *synthetic intrusion* yang positif | **J** | uji hipotesis BENIGN + proyeksi ruang fitur lokal |

**Catatan penting tentang klaim utama.** Bagian G adalah uji yang bisa membatalkan klaim naskah.
Jika XGBoost menaikkan baseline pada kategori-kategori sulit sehingga *headroom* mengecil dan manfaat
SMOTE ikut mengecil, korelasi bertahan dan klaim menguat. Jika pola korelasi hilang, klaim harus
ditulis ulang. Kedua hasil dicetak apa adanya oleh bagian K — tidak ada penyaringan.

| Bagian | Isi | Perkiraan waktu |
|---|---|---|
| 0–3 | Setup, unduh, bersihkan, split | ±15 menit pertama kali, lalu ±20 detik dari cache |
| G | Eksperimen terkontrol, **dijalankan sepotong demi sepotong** | ±70 menit per potongan |
| H | Agregasi tujuan false negative | < 2 menit |
| I | Ukuran kompleksitas + eksperimen representasi | ±30 menit |
| J | Uji hipotesis titik sintetis + proyeksi | ±15 menit |
| K | Tabel dan gambar siap naskah | < 2 menit |

**Dua hal yang membuat notebook ini tahan terhadap Colab yang terputus.** Pertama, data bersih
disimpan ke `clean_data.npz` di Drive, jadi tahap unduh dan pembersihan hanya berjalan sekali.
Kedua, bagian G dibagi menjadi potongan-potongan yang masing-masing selesai dalam satu sampai dua
jam, dengan batas waktu yang menghentikan pekerjaan secara rapi sebelum sesi mati. Sel **G0**
mencetak peta kemajuan dan menyarankan potongan berikutnya.


## 0. Pengaturan

In [ ]:
# ====== PENGATURAN (boleh diubah) ======
SEEDS           = [42, 43, 44]  # jumlah seed untuk noise floor; mulai dengan [42] jika ingin cepat
N_FOLDS         = 5             # stratified k-fold
CAP             = 30000         # batas maksimum record per kategori (sesuai naskah)
TARGET          = 24000         # target oversampling per kelas (sesuai naskah)
N_CONTROL       = 29            # titik kontrol (jumlah sampel latih Infiltration)
RF_TREES        = 100           # jumlah pohon Random Forest
RUN_ALL_METHODS = True          # semua konfigurasi Tabel 4
RANDOM_STATE    = 42
TEST_MODE       = False         # biarkan False (hanya untuk uji internal)

RUN_A_TABLE1    = True          # Tabel 1 + Tabel 6
RUN_B_SHAPE     = True          # bentuk distribusi 6 kategori
RUN_C_CONTROLLED= True          # eksperimen terkontrol v2
RUN_D_METHODS   = True          # Tabel 4 lengkap
RUN_E_CSE2018   = True          # kurva belajar CSE-CIC-IDS2018
TARGET_CATS     = ["Botnet", "PortScan", "Web Attack", "Infiltration", "DoS", "DDoS"]

# ====== TAMBAHAN v3 ======
# --- matriks LENGKAP yang akan dilaporkan di naskah (dibaca bagian H dan K) ---
MODELS          = ["rf", "xgb", "mlp"]   # keluarga model
METHODS_G       = ["base", "smote", "ros"]

# --- potongan yang DILATIH pada sesi ini (bagian G saja) -------------------
# Satu sesi Colab = satu potong. Ubah tiga baris ini tiap sesi; sisanya
# dilewati karena sudah ada di checkpoint. Sel G0 mencetak potongan berikutnya.
RUN_SEEDS       = [42]                   # lalu [43], lalu [44]
RUN_MODELS      = ["xgb"]                # "rf" sudah selesai di v2; "mlp" menyusul
RUN_METHODS     = ["base", "smote"]      # ROS untuk model baru tidak diminta editor
TIME_BUDGET_MIN = 150                    # berhenti rapi sebelum Colab memutus (0 = tanpa batas)
USE_CLEAN_CACHE = True                   # simpan data bersih ke Drive, lewati unduh saat mengulang
XGB_TREES       = 300
XGB_DEPTH       = 6
XGB_LR          = 0.1
MLP_HIDDEN      = (128, 64)
MLP_ITER        = 200

TOPK_FEATURES   = [5, 10, 20]            # eksperimen representasi
MODELS_REPR     = ["rf"]                 # cukup satu model agar hemat waktu
N_FOLDS_REPR    = 2                      # jumlah fold untuk eksperimen representasi
N_COMPLEX_CAP   = 3000                   # subsampel kelas lain untuk MST (N1)
N_SYNTH_J       = 4000                   # jumlah titik sintetis pada uji hipotesis

RUN_G_MODELS    = True                   # bagian G: tiga keluarga model
RUN_H_DEST      = True                   # bagian H: tujuan false negative
RUN_I_OVERLAP   = True                   # bagian I: ukuran kompleksitas
RUN_I_REPR      = True                   # bagian I: eksperimen representasi
RUN_J_SYNTH     = True                   # bagian J: uji hipotesis titik sintetis

import os, time, json, math, zipfile, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/juti_rerun"
except Exception:
    OUTDIR = "./juti_rerun"
os.makedirs(OUTDIR, exist_ok=True)
DATADIR = "/content/data" if os.path.isdir("/content") else "./data"
os.makedirs(DATADIR, exist_ok=True)
print("Hasil akan disimpan di:", OUTDIR)

In [ ]:
# Pasang library dengan versi yang tercatat di naskah (Sec. 3.6)
import subprocess, sys
if not TEST_MODE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn==1.6.1", "imbalanced-learn==0.14.2", "xgboost", "boto3"], check=False)
import sklearn, imblearn
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__)
print("Jika versi di atas belum 1.6.1 / 0.14.2: menu Runtime > Restart session, lalu jalankan ulang dari atas.")
import xgboost; print("xgboost", xgboost.__version__)

### 0b. Cache data bersih

Tahap unduh dan pembersihan memakan ±15 menit, dan Colab mengulanginya setiap kali runtime
terputus. Sel ini menyimpan hasilnya sekali ke Drive (`clean_data.npz`, sekitar 60 MB) lalu
membacanya kembali dalam hitungan detik pada sesi-sesi berikutnya.

Jalankan sel ini **sebelum** sel unduh. Kalau cache ditemukan, sel 1 dan 2 akan melewatkan
pekerjaannya sendiri. Hapus `clean_data.npz` kalau ingin membersihkan data dari awal.


In [ ]:
CLEAN_NPZ = os.path.join(OUTDIR, "clean_data.npz")
CACHE_OK = False
if USE_CLEAN_CACHE and os.path.exists(CLEAN_NPZ):
    try:
        z = np.load(CLEAN_NPZ, allow_pickle=True)
        X_all    = z["X_all"]
        y_all    = z["y_all"].astype(str)
        FEATURES = list(z["features"])
        CLASSES  = sorted(np.unique(y_all))
        CACHE_OK = True
        print(f"Cache ditemukan: {X_all.shape[0]:,} record x {X_all.shape[1]} fitur "
              f"— tahap unduh dan pembersihan dilewati.")
        print(pd.Series(y_all).value_counts().to_string())
    except Exception as e:
        print("Cache gagal dibaca, akan dibersihkan ulang:", e)
        CACHE_OK = False
else:
    print("Belum ada cache — sel unduh dan pembersihan akan berjalan seperti biasa.")

## 1. Unduh CIC-IDS2017

In [ ]:
if CACHE_OK:
    print("Cache dipakai — tahap unduh dilewati.")
else:
    ZIP_PATH = os.path.join(DATADIR, "MachineLearningCSV.zip")
    SOURCES = [
        "http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/MachineLearningCSV.zip",
        "https://huggingface.co/datasets/Mireu-Lab/CIC-IDS/resolve/main/ZIP/MachineLearningCSV.zip",
    ]
    drive_copy = os.path.join(OUTDIR, "MachineLearningCSV.zip")

    def zip_ok(path):
        try:
            with zipfile.ZipFile(path) as z:
                return any(n.endswith(".csv") for n in z.namelist())
        except Exception:
            return False

    if not TEST_MODE:
        if not zip_ok(ZIP_PATH) and zip_ok(drive_copy):
            import shutil; shutil.copy(drive_copy, ZIP_PATH)
        for url in SOURCES:
            if zip_ok(ZIP_PATH):
                break
            print("Mengunduh dari", url)
            subprocess.run(["wget", "-q", "--tries=2", "--timeout=60", "-O", ZIP_PATH, url])
        if not zip_ok(ZIP_PATH):
            raise RuntimeError("Unduhan gagal. Unggah MachineLearningCSV.zip ke Drive/juti_rerun lalu jalankan ulang sel ini.")
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(DATADIR)
    CSV_FILES = sorted(glob.glob(os.path.join(DATADIR, "**", "*.csv"), recursive=True))
    print(len(CSV_FILES), "file CSV ditemukan")
    for f in CSV_FILES: print(" -", os.path.basename(f))

## 2. Pembersihan data

In [ ]:
if CACHE_OK:
    print("Cache dipakai — tahap pembersihan dilewati.")
else:
    def category_of(label):
        l = str(label).strip()
        if l == "BENIGN": return "BENIGN"
        if l == "DDoS": return "DDoS"
        if l.startswith("DoS"): return "DoS"
        if l == "PortScan": return "PortScan"
        if l.startswith("Web Attack"): return "Web Attack"
        if l == "Bot": return "Botnet"
        if l == "Infiltration": return "Infiltration"
        if l in ("FTP-Patator", "SSH-Patator"): return "Brute Force"
        if l == "Heartbleed": return "Heartbleed"
        return None

    frames = []
    for f in CSV_FILES:
        d = pd.read_csv(f, encoding="latin-1", low_memory=False)
        d.columns = [c.strip() for c in d.columns]
        frames.append(d)
    raw = pd.concat(frames, ignore_index=True)
    print("Record mentah:", f"{len(raw):,}")

    raw = raw.drop_duplicates()
    print("Setelah hapus duplikat:", f"{len(raw):,}")

    label = raw["Label"]
    X = raw.drop(columns=["Label"])
    del raw, frames
    obj_cols = X.select_dtypes(include="object").columns
    for c_ in obj_cols:                      # kolom non-numerik diubah; yang gagal jadi null
        X[c_] = pd.to_numeric(X[c_], errors="coerce")
    X = X.loc[:, X.notna().any()]            # buang kolom yang seluruhnya non-numerik
    X = X.replace([np.inf, -np.inf], np.nan)
    keep = X.notna().all(axis=1)
    X, label = X[keep], label[keep]
    print("Setelah hapus baris inf/null:", f"{len(X):,}")

    cat = label.map(category_of)
    print("\nJumlah per label asli yang dibuang:")
    print(cat[cat.isin(["Brute Force", "Heartbleed"])].value_counts().to_string())
    mask = cat.isin(["BENIGN", "DoS", "DDoS", "PortScan", "Web Attack", "Botnet", "Infiltration"])
    X, cat = X[mask].reset_index(drop=True), cat[mask].reset_index(drop=True)

    const_cols = [c for c in X.columns if X[c].nunique() <= 1]
    X = X.drop(columns=const_cols)
    col_hash = {c_: pd.util.hash_pandas_object(X[c_], index=False).values.sum() for c_ in X.columns}
    seen, dup_cols = {}, []
    for c_ in X.columns:                     # kolom yang isinya identik dengan kolom sebelumnya
        h = col_hash[c_]
        if h in seen and X[c_].equals(X[seen[h]]):
            dup_cols.append(c_)
        else:
            seen.setdefault(h, c_)
    X = X.drop(columns=dup_cols)
    print("\nKolom konstan dihapus  (naskah: 8):", len(const_cols), const_cols)
    print("Kolom kembar dihapus :", dup_cols)
    print("Jumlah fitur akhir    (naskah: 65):", X.shape[1])

    print("\nJumlah per kategori sebelum dibatasi:")
    print(cat.value_counts().to_string())

    rng_cap = np.random.RandomState(RANDOM_STATE)
    idx = []
    for c, grp in cat.groupby(cat):
        ids = grp.index.values
        if len(ids) > CAP:
            ids = rng_cap.choice(ids, size=CAP, replace=False)
        idx.extend(ids)
    idx = np.sort(np.array(idx))
    X, cat = X.loc[idx].reset_index(drop=True), cat.loc[idx].reset_index(drop=True)
    print("\nSetelah dibatasi (naskah total: 124.127):", f"{len(X):,}")
    print(cat.value_counts().to_string())

    FEATURES = X.columns.tolist()
    X_all = X.values.astype(np.float64)
    y_all = cat.values.astype(str)
    CLASSES = sorted(np.unique(y_all))
    pd.DataFrame({"n": cat.value_counts()}).to_csv(os.path.join(OUTDIR, "counts_after_cap.csv"))

    summary_pre = {"n_raw_after_dedup_and_inf": int(len(X)), "n_features": len(FEATURES),
                   "constant_columns": const_cols, "duplicate_columns": dup_cols,
                   "counts_after_cap": cat.value_counts().to_dict(), "total_after_cap": int(len(X_all))}
    json.dump(summary_pre, open(os.path.join(OUTDIR, "preprocessing_summary.json"), "w"), indent=2)

if USE_CLEAN_CACHE and not CACHE_OK:
    np.savez_compressed(CLEAN_NPZ, X_all=X_all, y_all=np.asarray(y_all, dtype=object).astype(str),
                        features=np.array(FEATURES, dtype=object))
    print("\nData bersih disimpan ke cache:", CLEAN_NPZ)
    print("Sesi berikutnya tidak perlu mengunduh dan membersihkan ulang.")

## 3. Pembagian 80:20 dan standardisasi

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(X_tr_raw)          # hanya dari data latih
X_tr = scaler.transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)
print("Latih:", X_tr.shape, "| Uji:", X_te.shape)
print("Infiltration latih/uji:", (y_tr == "Infiltration").sum(), "/", (y_te == "Infiltration").sum())

## G. Eksperimen terkontrol dengan tiga keluarga model

Permintaan editor nomor 1. Desainnya persis sama dengan bagian C di v2 — setiap kategori dipangkas ke
n = 29 sampel latih, lalu dibandingkan base / SMOTE / ROS — tetapi sekarang dijalankan untuk tiga
keluarga model:

* **RF** — Random Forest, sama seperti naskah sekarang. Baris yang sudah ada di `secC_controlled_v2.csv`
  dibaca ulang dan diberi label `model = "rf"`, jadi tidak dilatih ulang.
* **XGB** — XGBoost dengan `hist` tree method. Inilah baseline kuat yang diminta editor.
* **MLP** — jaringan saraf dua lapis (128, 64). Ini menjawab bagian "stronger neural network backbones"
  dari permintaan yang sama, dan harganya murah karena datanya sudah terstandardisasi.

Satu hal yang perlu diperhatikan: XGBoost memerlukan label numerik, dan pemetaan label dibuat
**sekali** dari `CLASSES` sehingga indeks kelas konsisten di seluruh fold.

### Bagian ini dijalankan sepotong demi sepotong

Matriks lengkapnya 810 model, jauh melebihi umur satu sesi Colab. Karena itu bagian G tidak
dijalankan sekaligus. Tiga saklar di sel pengaturan menentukan potongan mana yang dilatih
sekarang; sisanya dilewati karena sudah ada di checkpoint:

```
RUN_SEEDS   = [42]                 # satu seed per sesi
RUN_MODELS  = ["xgb"]              # satu keluarga model per sesi
RUN_METHODS = ["base", "smote"]
```

Ditambah `TIME_BUDGET_MIN`, yang memeriksa jam **sebelum** setiap model dimulai dan berhenti
rapi ketika batas tercapai — jadi sesi tidak pernah mati di tengah pelatihan dan kehilangan
model yang sedang dikerjakan.

Urutan yang disarankan, masing-masing sekitar satu sampai dua jam:

| Sesi | RUN_SEEDS | RUN_MODELS | RUN_METHODS | Model | Perkiraan |
|---|---|---|---|---|---|
| 1 | `[42]` | `["xgb"]` | `["base","smote"]` | 60 | ±70 menit |
| 2 | `[43]` | `["xgb"]` | `["base","smote"]` | 60 | ±70 menit |
| 3 | `[44]` | `["xgb"]` | `["base","smote"]` | 60 | ±70 menit |
| 4–6 | tiap seed | `["mlp"]` | `["base","smote"]` | 60 | ±20 menit |

Setelah sesi 3, bagian H sampai K sudah bisa dijalankan dan menghasilkan seluruh tabel untuk
RF dan XGBoost. MLP di sesi 4–6 adalah pelengkap: surat editor menulis "XGBoost **atau**
backbone neural", jadi XGBoost sendiri sudah menjawab permintaannya.

**ROS tidak diulang untuk model baru.** Argumen SMOTE-versus-duplikasi sudah tegak dengan
Random Forest di v2, dan editor tidak memintanya lagi. Menghilangkan ROS dari model baru
memangkas sepertiga pekerjaan. Kalau nanti ingin dilengkapi, cukup tambahkan `"ros"` ke
`RUN_METHODS` dan jalankan lagi.


### G0. Peta kemajuan dan potongan berikutnya

Sel ini **tidak melatih apa pun**. Ia membaca checkpoint, mencetak berapa model yang sudah
selesai per seed dan per keluarga model, memperkirakan sisa waktu dari waktu latih yang
benar-benar terukur, lalu menyarankan isi `RUN_SEEDS` / `RUN_MODELS` / `RUN_METHODS` untuk sesi
berikutnya. Aman dijalankan kapan saja, termasuk di tengah pekerjaan.


In [ ]:
CKG = os.path.join(OUTDIR, "secG_controlled_v3.csv")
COLS_G = ["seed", "fold", "category", "model", "method", "n_train",
          "tp", "fp", "fn", "fn_dest", "fit_seconds"]
d = pd.read_csv(CKG) if os.path.exists(CKG) else pd.DataFrame(columns=COLS_G)
if "fit_seconds" not in d.columns:
    d["fit_seconds"] = np.nan

done = set(zip(d.seed, d.fold, d.category, d.model, d.method))
plan = [(s, f, c, mo, me) for s in SEEDS for f in range(N_FOLDS)
        for c in TARGET_CATS for mo in MODELS for me in METHODS_G]
pending = [k for k in plan if k not in done]

print(f"Matriks lengkap : {len(plan)} model")
print(f"Sudah selesai   : {len(plan) - len(pending)}")
print(f"Belum           : {len(pending)}\n")

grid = pd.DataFrame(
    [[sum(1 for k in plan if k[0] == s and k[3] == mo and k[4] == me and k in done)
      for me in METHODS_G for mo in MODELS] for s in SEEDS],
    index=[f"seed {s}" for s in SEEDS],
    columns=pd.MultiIndex.from_product([METHODS_G, MODELS]))
per_cell = N_FOLDS * len(TARGET_CATS)
print(f"Model selesai per sel (maksimum {per_cell}):")
print(grid.to_string(), "\n")

# perkiraan sisa waktu dari waktu latih yang terukur
med = d.dropna(subset=["fit_seconds"]).groupby("model")["fit_seconds"].median()
default_s = {"rf": 85.0, "xgb": 71.0, "mlp": 20.0}
if len(med):
    print("Waktu latih terukur (median, detik):",
          ", ".join(f"{m}={v:.0f}" for m, v in med.items()))
est = sum(float(med.get(k[3], default_s.get(k[3], 60.0))) for k in pending)
print(f"Perkiraan sisa seluruh matriks: {est/3600:.1f} jam\n")

if not pending:
    print("Bagian G sudah lengkap. Lanjutkan ke bagian H.")
else:
    # Prioritas potongan mengikuti kepentingan ilmiahnya, bukan biaya komputasinya.
    # XGBoost adalah yang diminta editor, jadi ia didahulukan; ROS untuk model baru
    # hanya pelengkap dan dikerjakan paling akhir.
    PRIO_MODEL  = {"xgb": 0, "mlp": 1, "rf": 2}
    PRIO_METHOD = {"base": 0, "smote": 0, "ros": 1}      # base dan smote satu paket
    rank = lambda k: (PRIO_METHOD.get(k[4], 9), PRIO_MODEL.get(k[3], 9), k[0])
    best = min(pending, key=rank)
    tier, nxt_model, nxt_seed = PRIO_METHOD.get(best[4], 9), best[3], best[0]
    nxt_methods = sorted({k[4] for k in pending if k[0] == nxt_seed and k[3] == nxt_model
                          and PRIO_METHOD.get(k[4], 9) == tier})
    sl = [k for k in pending if k[0] == nxt_seed and k[3] == nxt_model and k[4] in nxt_methods]
    t_slice = len(sl) * float(med.get(nxt_model, default_s.get(nxt_model, 60.0)))
    print("Potongan berikutnya — salin ke sel pengaturan lalu jalankan ulang dari atas:")
    print(f"    RUN_SEEDS   = [{nxt_seed}]")
    print(f'    RUN_MODELS  = ["{nxt_model}"]')
    print(f"    RUN_METHODS = {nxt_methods}")
    print(f"\n  {len(sl)} model, perkiraan {t_slice/60:.0f} menit "
          f"(batas waktu sesi sekarang: {TIME_BUDGET_MIN} menit)")
    core = [k for k in pending if PRIO_METHOD.get(k[4], 9) == 0 and k[3] in ("rf", "xgb")]
    if not core:
        print("\n  Catatan: RF dan XGBoost sudah lengkap untuk base dan SMOTE — "
              "bagian H sampai K sudah bisa dijalankan sekarang. Sisanya pelengkap.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE, RandomOverSampler
from xgboost import XGBClassifier

# --- Geometri subsampel (identik dengan bagian B/C di v2) -------------------
# Didefinisikan ulang di sini supaya v3 bisa berdiri sendiri: kalau
# secC_subsample_shape.csv tidak ada di folder Drive, file itu dibangun ulang
# di dalam loop di bawah, pada subsampel yang persis sama. Tanpa file ini,
# bagian K tidak bisa menghitung korelasi dispersi dan intrusi.
K_NN = 5

def example_types(X_min, X_other, k=K_NN):
    X_both = np.vstack([X_min, X_other])
    is_min = np.r_[np.ones(len(X_min), bool), np.zeros(len(X_other), bool)]
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_both).kneighbors(X_min)
    same = is_min[ind[:, 1:]].sum(axis=1)
    return {"safe (%)": np.mean(same >= 4) * 100,
            "borderline (%)": np.mean((same == 2) | (same == 3)) * 100,
            "rare (%)": np.mean(same == 1) * 100,
            "outlier (%)": np.mean(same == 0) * 100}

def dispersion_ratio(X_min, nn_other):
    d_in, _ = NearestNeighbors(n_neighbors=2).fit(X_min).kneighbors(X_min)
    d_out, _ = nn_other.kneighbors(X_min, n_neighbors=1)
    return d_in[:, 1].mean() / max(d_out[:, 0].mean(), 1e-12)

def synthetic_intrusion(X_min, nn_other, rng, k=K_NN, n_new=2000):
    k = min(k, len(X_min) - 1)
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    i = rng.integers(0, len(X_min), n_new)
    j = ind[i, rng.integers(1, k + 1, n_new)]
    lam = rng.random((n_new, 1))
    X_new = X_min[i] + lam * (X_min[j] - X_min[i])          # persamaan (1)
    d_min, _ = NearestNeighbors(n_neighbors=1).fit(X_min).kneighbors(X_new)
    d_oth, _ = nn_other.kneighbors(X_new, n_neighbors=1)
    return np.mean(d_oth[:, 0] < d_min[:, 0]) * 100

CLS2ID = {c: i for i, c in enumerate(CLASSES)}
ID2CLS = np.array(CLASSES)

def make_model(name, seed):
    if name == "rf":
        return RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=seed)
    if name == "xgb":
        return XGBClassifier(n_estimators=XGB_TREES, max_depth=XGB_DEPTH, learning_rate=XGB_LR,
                             subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                             objective="multi:softmax", num_class=len(CLASSES),
                             n_jobs=-1, random_state=seed, verbosity=0)
    if name == "mlp":
        return MLPClassifier(hidden_layer_sizes=MLP_HIDDEN, max_iter=MLP_ITER,
                             early_stopping=True, n_iter_no_change=10,
                             random_state=seed)
    raise ValueError(name)

def fit_predict(name, seed, Xs, ys, Xte):
    """Melatih dan memprediksi; label selalu kembali sebagai string nama kategori.

    XGBoost memang memerlukan label numerik. MLPClassifier juga dilatih dengan label
    numerik: dengan early_stopping aktif, sebagian versi scikit-learn gagal menilai
    himpunan validasi bila labelnya bertipe string."""
    m = make_model(name, seed)
    if name in ("xgb", "mlp"):
        m.fit(Xs, np.array([CLS2ID[v] for v in ys], dtype=int))
        return ID2CLS[np.asarray(m.predict(Xte)).astype(int)]
    m.fit(Xs, ys)
    return np.asarray(m.predict(Xte)).astype(str)

def smote_strategy(y):
    vc = pd.Series(y).value_counts()
    return {c: TARGET for c, n in vc.items() if n < TARGET}

def counts_for(y_true, y_pred, c):
    return (int(((y_pred == c) & (y_true == c)).sum()),
            int(((y_pred == c) & (y_true != c)).sum()),
            int(((y_pred != c) & (y_true == c)).sum()))

CKC  = os.path.join(OUTDIR, "secC_controlled_v2.csv")      # hasil v2 (hanya RF)
CKS  = os.path.join(OUTDIR, "secC_subsample_shape.csv")    # geometri subsampel, dipakai lagi di bagian J
CKG  = os.path.join(OUTDIR, "secG_controlled_v3.csv")      # hasil v3 (semua model)
COLS_G = ["seed", "fold", "category", "model", "method", "n_train",
          "tp", "fp", "fn", "fn_dest", "fit_seconds"]

if os.path.exists(CKG):
    doneG = pd.read_csv(CKG)
else:
    doneG = pd.DataFrame(columns=COLS_G)
    if os.path.exists(CKC):                                 # impor hasil RF dari v2 tanpa melatih ulang
        v2 = pd.read_csv(CKC)
        v2["model"] = "rf"
        doneG = v2.reindex(columns=COLS_G)
        doneG.to_csv(CKG, index=False)
        print(f"Diimpor {len(doneG)} baris RF dari v2 — tidak dilatih ulang.")

doneS = pd.read_csv(CKS) if os.path.exists(CKS) else pd.DataFrame(
    columns=["seed", "fold", "category", "safe (%)", "borderline (%)", "rare (%)",
             "outlier (%)", "dispersion ratio", "synthetic intrusion (%)"])
keysS = set(zip(doneS.seed, doneS.fold, doneS.category)) if len(doneS) else set()
if len(doneS):
    print(f"Geometri subsampel: {len(doneS)} baris sudah ada, dipakai ulang.")
else:
    print("Geometri subsampel belum ada — akan dibangun ulang sambil melatih model.")

if "fit_seconds" not in doneG.columns:
    doneG["fit_seconds"] = np.nan
keysG = set(zip(doneG.seed, doneG.fold, doneG.category, doneG.model, doneG.method))

# Potongan yang dikerjakan sesi ini. Matriks lengkap tetap MODELS x METHODS_G;
# RUN_* hanya memilih bagian mana yang dilatih sekarang.
SLICE = [(s, f, c, mo, me) for s in RUN_SEEDS for f in range(N_FOLDS)
         for c in TARGET_CATS for mo in RUN_MODELS for me in RUN_METHODS]
todo_slice = [k for k in SLICE if k not in keysG]
print(f"Potongan sesi ini : seed {RUN_SEEDS}, model {RUN_MODELS}, metode {RUN_METHODS}")
print(f"Perlu dilatih     : {len(todo_slice)} dari {len(SLICE)} model")
if TIME_BUDGET_MIN:
    print(f"Batas waktu sesi  : {TIME_BUDGET_MIN} menit "
          f"(berhenti rapi, checkpoint tetap tersimpan)")

t_start, n_run, stopped = time.time(), 0, False
for seed in (RUN_SEEDS if RUN_G_MODELS else []):
    if stopped:
        break
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        if stopped:
            break
        need = any((seed, fold, c, mo, me) not in keysG
                   for c in TARGET_CATS for mo in RUN_MODELS for me in RUN_METHODS) or \
               any((seed, fold, c) not in keysS for c in TARGET_CATS)
        if not need:
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for c in TARGET_CATS:
            if stopped:
                break
            rng_t = np.random.default_rng(seed * 100 + fold)   # subsampel identik dengan v1/v2
            idx_c = np.where(ytr_f == c)[0]
            if len(idx_c) > N_CONTROL:
                idx_c = rng_t.choice(idx_c, size=N_CONTROL, replace=False)
            keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
            Xk, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
            if (seed, fold, c) not in keysS:          # bangun ulang geometri bila belum ada
                X_min, X_oth = Xtr_f[idx_c], Xtr_f[ytr_f != c]
                nn_o = NearestNeighbors(n_neighbors=1).fit(X_oth)
                srow = dict(seed=seed, fold=fold, category=c, **example_types(X_min, X_oth))
                srow["dispersion ratio"] = dispersion_ratio(X_min, nn_o)
                srow["synthetic intrusion (%)"] = synthetic_intrusion(
                    X_min, nn_o, np.random.default_rng(seed + fold))   # seed identik dengan v2
                doneS = pd.concat([doneS, pd.DataFrame([srow])], ignore_index=True)
                doneS.to_csv(CKS, index=False); keysS.add((seed, fold, c))
            # oversampling dihitung sekali per (fold, kategori, metode), dipakai semua model
            for me in RUN_METHODS:
                if stopped:
                    break
                todo = [mo for mo in RUN_MODELS if (seed, fold, c, mo, me) not in keysG]
                if not todo:
                    continue
                if me == "smote":
                    Xs, ys = SMOTE(sampling_strategy=smote_strategy(yk), k_neighbors=5,
                                   random_state=seed).fit_resample(Xk, yk)
                elif me == "ros":
                    Xs, ys = RandomOverSampler(sampling_strategy=smote_strategy(yk),
                                               random_state=seed).fit_resample(Xk, yk)
                else:
                    Xs, ys = Xk, yk
                for mo in todo:
                    # batas waktu diperiksa SEBELUM model dimulai, supaya sesi tidak
                    # pernah terputus di tengah pelatihan
                    if TIME_BUDGET_MIN and (time.time() - t_start) / 60 >= TIME_BUDGET_MIN:
                        stopped = True
                        break
                    t0 = time.time()
                    pred = fit_predict(mo, seed, Xs, ys, Xte_f)
                    dt = time.time() - t0
                    tp, fp, fn = counts_for(yte_f, pred, c)
                    dest = pd.Series(pred[(yte_f == c) & (pred != c)]).value_counts().to_dict()
                    row = dict(seed=seed, fold=fold, category=c, model=mo, method=me,
                               n_train=len(idx_c), tp=tp, fp=fp, fn=fn,
                               fn_dest=json.dumps({str(k): int(v) for k, v in dest.items()}),
                               fit_seconds=round(dt, 1))
                    doneG = pd.concat([doneG, pd.DataFrame([row])], ignore_index=True)
                    doneG.to_csv(CKG, index=False)
                    keysG.add((seed, fold, c, mo, me)); n_run += 1
                    left = len([k for k in SLICE if k not in keysG])
                    avg = (time.time() - t_start) / n_run
                    print(f"seed {seed} fold {fold} {c:12s} {mo:3s} {me:5s} {dt:5.0f} s "
                          f"| sisa potongan {left} ≈ {left*avg/60:.0f} menit")

left = len([k for k in SLICE if k not in keysG])
if stopped:
    print(f"\nBerhenti karena batas waktu {TIME_BUDGET_MIN} menit tercapai. "
          f"Semua hasil sudah tersimpan.")
    print(f"Sisa potongan ini: {left} model. Jalankan ulang notebook dengan RUN_* yang sama "
          f"untuk melanjutkan — yang sudah selesai akan dilewati.")
elif left == 0:
    print(f"\nPotongan ini selesai. Jalankan sel G0 untuk melihat potongan berikutnya.")
else:
    print(f"\nBagian G berhenti dengan {left} model tersisa pada potongan ini.")

## H. Tujuan false negative untuk semua kategori dan semua model

Permintaan editor nomor 3. Datanya sudah direkam di kolom `fn_dest` — tinggal diagregasi.

Yang menarik bukan sekadar berapa banyak serangan yang terlewat, melainkan **ke mana** serangan itu
jatuh. Ada dua kemungkinan yang konsekuensinya berbeda bagi perancang pipeline NIDS:

* serangan jatuh ke **BENIGN** — ini kegagalan yang sesungguhnya, serangan lolos tanpa jejak;
* serangan jatuh ke **kelas serangan lain** — alarm tetap berbunyi, hanya labelnya salah.

Kolom `to_benign (%)` di bawah inilah angka yang paling berguna untuk praktik, dan sejauh yang kami
tahu angka ini belum pernah dilaporkan secara sistematis per kategori dalam literatur SMOTE untuk NIDS.


### H0. Melengkapi tujuan kesalahan Random Forest

Baris Random Forest untuk kondisi `base` dan `smote` diwarisi dari notebook v1 lewat rantai impor
v1 → v2 → v3, dan v1 belum merekam kelas prediksi dari setiap false negative. Akibatnya kolom
`fn_dest` kosong untuk sebagian baris RF, dan Tabel 10 di naskah hanya bisa memuat XGBoost dan MLP.

Sel ini melatih ulang **hanya** konfigurasi yang kolom `fn_dest`-nya kosong, memakai subsampel dan
benih yang persis sama, lalu:

1. membandingkan `tp`, `fp`, `fn` hasil latih ulang dengan nilai yang sudah tersimpan;
2. bila cocok, mengisi `fn_dest` dan membiarkan angka lama apa adanya;
3. bila **tidak** cocok, mencatatnya sebagai ketidaksesuaian dan **tidak** menimpa apa pun.

Langkah kedua itu sekaligus uji reproduksibilitas: kalau angka lama tidak bisa dihasilkan ulang,
itu pertanda versi pustaka sudah bergeser dan seluruh kolom RF perlu dijalankan ulang, bukan
sekadar ditambal. Jangan ubah `OVERWRITE_ON_MISMATCH` menjadi `True` tanpa memahami akibatnya:
angka di Tabel 4, 5, 6, 7 dan 10 ikut berubah dan naskah harus menyesuaikan.

Perkiraan waktu: sekitar 85 detik per model. Jumlah model yang perlu dilatih dicetak di awal, dan
`TIME_BUDGET_MIN` berlaku di sini juga, jadi sel ini aman dijalankan sepotong demi sepotong.


In [ ]:
OVERWRITE_ON_MISMATCH = False     # biarkan False kecuali Anda tahu persis akibatnya

CKG = os.path.join(OUTDIR, "secG_controlled_v3.csv")
d = pd.read_csv(CKG)
if "fit_seconds" not in d.columns:
    d["fit_seconds"] = np.nan
if "fn_dest" not in d.columns:
    d["fn_dest"] = ""

def _empty(v):
    return (not isinstance(v, str)) or (v.strip() in ("", "{}", "nan"))

need = d[(d.model == "rf") & d.fn_dest.map(_empty)].copy()
print(f"Baris RF tanpa fn_dest: {len(need)}")
if len(need):
    print(need.groupby(["category", "method"]).size().to_string())
    print(f"Perkiraan waktu: {len(need) * 85 / 3600:.1f} jam\n")
else:
    print("Tidak ada yang perlu dilengkapi.")

todo = {(int(r.seed), int(r.fold)): None for r in need.itertuples()}
t_start, n_run, stopped = time.time(), 0, False
filled, mismatch = 0, []

for seed in sorted({k[0] for k in todo}):
    if stopped:
        break
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        if stopped:
            break
        sub = need[(need.seed == seed) & (need.fold == fold)]
        if not len(sub):
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for c in sorted(set(sub.category)):
            if stopped:
                break
            rng_t = np.random.default_rng(seed * 100 + fold)   # subsampel identik dengan v1/v2/v3
            idx_c = np.where(ytr_f == c)[0]
            if len(idx_c) > N_CONTROL:
                idx_c = rng_t.choice(idx_c, size=N_CONTROL, replace=False)
            keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
            Xk, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
            for me in sorted(set(sub[sub.category == c].method)):
                if TIME_BUDGET_MIN and (time.time() - t_start) / 60 >= TIME_BUDGET_MIN:
                    stopped = True
                    break
                t0 = time.time()
                if me == "smote":
                    Xs, ys = SMOTE(sampling_strategy=smote_strategy(yk), k_neighbors=5,
                                   random_state=seed).fit_resample(Xk, yk)
                elif me == "ros":
                    Xs, ys = RandomOverSampler(sampling_strategy=smote_strategy(yk),
                                               random_state=seed).fit_resample(Xk, yk)
                else:
                    Xs, ys = Xk, yk
                pred = fit_predict("rf", seed, Xs, ys, Xte_f)
                tp, fp, fn = counts_for(yte_f, pred, c)
                dest = pd.Series(pred[(yte_f == c) & (pred != c)]).value_counts().to_dict()
                dest = json.dumps({str(k): int(v) for k, v in dest.items()})

                m = (d.seed == seed) & (d.fold == fold) & (d.category == c) & \
                    (d.model == "rf") & (d.method == me)
                old = d.loc[m, ["tp", "fp", "fn"]].iloc[0]
                same = (int(old.tp), int(old.fp), int(old.fn)) == (tp, fp, fn)
                if same or OVERWRITE_ON_MISMATCH:
                    d.loc[m, "fn_dest"] = dest
                    if not same:
                        d.loc[m, ["tp", "fp", "fn"]] = [tp, fp, fn]
                    filled += 1
                else:
                    mismatch.append(dict(seed=seed, fold=fold, category=c, method=me,
                                         stored=(int(old.tp), int(old.fp), int(old.fn)),
                                         recomputed=(tp, fp, fn)))
                d.to_csv(CKG, index=False)
                n_run += 1
                left = len(need) - n_run
                avg = (time.time() - t_start) / n_run
                flag = "cocok" if same else "TIDAK COCOK"
                print(f"seed {seed} fold {fold} {c:12s} rf {me:5s} {time.time()-t0:5.0f} s "
                      f"| {flag} | sisa {left} ≈ {left*avg/60:.0f} menit")

print(f"\nTerisi: {filled} | tidak cocok: {len(mismatch)}")
if mismatch:
    print("\nKETIDAKSESUAIAN — angka lama TIDAK ditimpa:")
    print(pd.DataFrame(mismatch).to_string(index=False))
    print("\nIni berarti hasil lama tidak dapat direproduksi pada lingkungan sekarang.")
    print("Jangan tambal; jalankan ulang seluruh kolom RF dengan menghapus baris RF dari")
    print("secG_controlled_v3.csv lalu menjalankan bagian G dengan RUN_MODELS = [\"rf\"].")
elif stopped:
    print("Berhenti karena batas waktu. Jalankan ulang sel ini untuk melanjutkan.")
elif filled:
    print("Selesai. Jalankan ulang bagian H untuk mendapatkan Tabel 10 yang lengkap,")
    print("lalu kirimkan secH_error_destination.csv yang baru.")

In [ ]:
dg = pd.read_csv(CKG)
dg = dg[dg.seed.isin(SEEDS) & dg.category.isin(TARGET_CATS) & dg.model.isin(MODELS)]

def dest_rows(df):
    out = []
    for _, r in df.iterrows():
        try:
            d = json.loads(r["fn_dest"]) if isinstance(r["fn_dest"], str) and r["fn_dest"].strip() else {}
        except Exception:
            d = {}
        for k, v in d.items():
            out.append(dict(seed=r["seed"], fold=r["fold"], category=r["category"],
                            model=r["model"], method=r["method"], dest=k, n=int(v)))
    return pd.DataFrame(out)

dest = dest_rows(dg)
if len(dest) == 0:
    print("Kolom fn_dest masih kosong — jalankan bagian G lebih dulu.")
else:
    tot = (dest.groupby(["category", "model", "method"])["n"].sum()
               .rename("fn_total").reset_index())
    ben = (dest[dest.dest == "BENIGN"].groupby(["category", "model", "method"])["n"].sum()
               .rename("fn_benign").reset_index())
    h = tot.merge(ben, how="left").fillna({"fn_benign": 0})
    h["to_benign (%)"] = 100 * h["fn_benign"] / h["fn_total"].clip(lower=1)
    # tujuan selain BENIGN yang paling sering
    oth = dest[dest.dest != "BENIGN"]
    top = (oth.sort_values("n", ascending=False)
              .groupby(["category", "model", "method"])
              .agg(top_other=("dest", "first")).reset_index())
    h = h.merge(top, how="left")
    h.to_csv(os.path.join(OUTDIR, "secH_error_destination.csv"), index=False)

    print("Tujuan false negative — seluruh kategori, seluruh model\n")
    for mo in MODELS:
        sub = h[h.model == mo]
        if not len(sub):
            continue
        piv = sub.pivot_table(index="category", columns="method", values="to_benign (%)")
        piv = piv.reindex(index=[c for c in TARGET_CATS if c in piv.index],
                          columns=[m for m in METHODS_G if m in piv.columns])
        print(f"--- model {mo.upper()}: persentase serangan terlewat yang jatuh ke BENIGN ---")
        print(piv.round(1).to_string(), "\n")

    print("--- tujuan non-BENIGN yang paling sering (model/metode digabung) ---")
    print(oth.groupby(["category", "dest"])["n"].sum().reset_index()
             .sort_values(["category", "n"], ascending=[True, False])
             .groupby("category").head(2).to_string(index=False))

## I. Tumpang tindih fitur dan eksperimen representasi

Permintaan editor nomor 2. Naskah sekarang menyebut tumpang tindih fitur sebagai dugaan penyebab
Infiltration mentok, tetapi tidak pernah mengukurnya. Bagian ini mengukurnya dengan empat ukuran
kompleksitas yang baku (Ho & Basu), dihitung satu-lawan-semua untuk setiap kategori:

* **F1 — Fisher discriminant ratio.** Nilai maksimum di antara seluruh fitur. Semakin kecil, semakin
  tidak ada satu pun fitur tunggal yang memisahkan kategori ini dari sisanya.
* **N1 — fraksi titik di batas kelas.** Dihitung dari minimum spanning tree: proporsi titik kategori
  yang bertetangga MST dengan titik kelas lain.
* **N2 — rasio jarak intra/inter kelas.** Semakin mendekati 1, semakin tidak terpisahkan.
* **N3 — galat leave-one-out 1-NN.** Ukuran paling langsung dari ketertumpangtindihan.

Lalu **eksperimen representasi**: perbandingan base/SMOTE diulang hanya pada `TOPK_FEATURES` fitur
paling diskriminatif menurut Fisher ratio. Kalau manfaat SMOTE berubah ketika representasinya
diperbaiki, yang membatasi memang ruang fitur, bukan jumlah sampel — persis pertanyaan editor.


In [ ]:
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import cdist

def fisher_ratio_per_feature(Xa, Xb):
    ma, mb = Xa.mean(0), Xb.mean(0)
    va, vb = Xa.var(0), Xb.var(0)
    return (ma - mb) ** 2 / np.maximum(va + vb, 1e-12)

def complexity_measures(X_min, X_oth, rng, cap=N_COMPLEX_CAP):
    """F1, N1, N2, N3 satu-lawan-semua. X_oth disubsampel agar MST tetap muat di memori."""
    if len(X_oth) > cap:
        X_oth = X_oth[rng.choice(len(X_oth), size=cap, replace=False)]
    fr = fisher_ratio_per_feature(X_min, X_oth)
    Xb = np.vstack([X_min, X_oth])
    lab = np.r_[np.ones(len(X_min), bool), np.zeros(len(X_oth), bool)]
    D = cdist(Xb, Xb)
    # N1: titik yang terhubung MST ke titik berlabel beda
    mst = minimum_spanning_tree(D).tocoo()
    cross = lab[mst.row] != lab[mst.col]
    touched = np.zeros(len(Xb), bool)
    touched[mst.row[cross]] = True
    touched[mst.col[cross]] = True
    n1 = touched.mean() * 100
    # N2: rata-rata jarak intra kelas / rata-rata jarak inter kelas, untuk titik minoritas
    Dm = D[:len(X_min)]
    d_in = np.sort(Dm[:, :len(X_min)], axis=1)[:, 1] if len(X_min) > 1 else np.zeros(len(X_min))
    d_out = Dm[:, len(X_min):].min(axis=1)
    n2 = d_in.sum() / max(d_out.sum(), 1e-12)
    # N3: galat 1-NN leave-one-out pada gabungan
    np.fill_diagonal(D, np.inf)
    n3 = (lab[D.argmin(axis=1)] != lab).mean() * 100
    return {"F1 (Fisher max)": float(fr.max()), "N1 (%)": float(n1),
            "N2": float(n2), "N3 (%)": float(n3)}, fr

CKI = os.path.join(OUTDIR, "secI_complexity.csv")
rng_i = np.random.default_rng(RANDOM_STATE)
rows_i, fisher_by_cat = [], {}

for c in (TARGET_CATS if RUN_I_OVERLAP else []):
    t0 = time.time()
    X_cat = X_tr[y_tr == c]
    pick = rng_i.choice(len(X_cat), size=min(N_CONTROL, len(X_cat)), replace=False)
    X_min = X_cat[pick]                       # kondisi n = 29, sama dengan eksperimen terkontrol
    X_oth = X_tr[y_tr != c]
    m, fr = complexity_measures(X_min, X_oth, rng_i)
    fisher_by_cat[c] = fr
    rows_i.append(dict(Category=c, n=len(X_min), **m))
    print(f"{c:13s} F1={m['F1 (Fisher max)']:7.3f}  N1={m['N1 (%)']:5.1f}%  "
          f"N2={m['N2']:6.3f}  N3={m['N3 (%)']:5.1f}%   ({time.time()-t0:.0f} s)")

if rows_i:
    comp = pd.DataFrame(rows_i)
    comp.to_csv(CKI, index=False)
    print("\n", comp.round(3).to_string(index=False))
    np.save(os.path.join(OUTDIR, "secI_fisher_per_feature.npy"),
            np.vstack([fisher_by_cat[c] for c in TARGET_CATS]))

In [ ]:
# --- Eksperimen representasi: base vs SMOTE hanya pada TOPK fitur paling diskriminatif ---
CKR = os.path.join(OUTDIR, "secI_representation.csv")
doneR = pd.read_csv(CKR) if os.path.exists(CKR) else pd.DataFrame(
    columns=["seed", "fold", "category", "model", "method", "k", "tp", "fp", "fn"])
keysR = set(zip(doneR.seed, doneR.fold, doneR.category, doneR.model, doneR.method, doneR.k))

for seed in (SEEDS if RUN_I_REPR else []):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        if fold >= N_FOLDS_REPR:                      # eksperimen representasi cukup sebagian fold
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for c in TARGET_CATS:
            rng_t = np.random.default_rng(seed * 100 + fold)
            idx_c = np.where(ytr_f == c)[0]
            if len(idx_c) > N_CONTROL:
                idx_c = rng_t.choice(idx_c, size=N_CONTROL, replace=False)
            keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
            Xk_full, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
            # peringkat fitur dihitung HANYA dari data latih fold ini
            fr = fisher_ratio_per_feature(Xtr_f[idx_c], Xtr_f[ytr_f != c])
            for k in TOPK_FEATURES:
                sel = np.argsort(fr)[::-1][:k]
                Xk, Xte_k = Xk_full[:, sel], Xte_f[:, sel]
                for me in ["base", "smote"]:
                    for mo in MODELS_REPR:
                        if (seed, fold, c, mo, me, k) in keysR:
                            continue
                        if me == "smote":
                            Xs, ys = SMOTE(sampling_strategy=smote_strategy(yk), k_neighbors=5,
                                           random_state=seed).fit_resample(Xk, yk)
                        else:
                            Xs, ys = Xk, yk
                        pred = fit_predict(mo, seed, Xs, ys, Xte_k)
                        tp, fp, fn = counts_for(yte_f, pred, c)
                        doneR = pd.concat([doneR, pd.DataFrame([dict(
                            seed=seed, fold=fold, category=c, model=mo, method=me,
                            k=k, tp=tp, fp=fp, fn=fn)])], ignore_index=True)
                        doneR.to_csv(CKR, index=False); keysR.add((seed, fold, c, mo, me, k))
            print(f"seed {seed} fold {fold} {c:12s} representasi selesai")
print("Eksperimen representasi selesai.")

## J. Mengapa *synthetic intrusion* berkorelasi positif

Permintaan editor nomor 4, dan satu-satunya bagian yang menguji sebuah **hipotesis**, bukan sekadar
melaporkan angka.

Dalam naskah, `synthetic intrusion rate` ι — proporsi titik SMOTE yang mendarat lebih dekat ke kelas
lain daripada ke kelas asalnya — berkorelasi **positif** dengan manfaat SMOTE. Secara teori arahnya
seharusnya terbalik: titik yang salah tempat mestinya merusak.

**Hipotesis yang diuji di sini:** titik sintetis yang salah tempat sebagian besar mendarat di wilayah
**BENIGN**, bukan di wilayah kelas serangan lain. Karena BENIGN adalah kelas yang paling mudah
dipisahkan dan paling padat, titik yang mendarat di sana tidak bersaing dengan kelas serangan mana pun;
efeknya lebih mirip menandai batas keputusan daripada mencemari kelas tetangga.

Hipotesis ini bisa salah, dan bentuk kesalahannya jelas: kalau titik salah tempat ternyata tersebar ke
kelas-kelas serangan, penjelasannya harus dicari di tempat lain. Yang dicetak di bawah adalah
distribusi tujuan apa adanya.


In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def synthetic_destination(X_min, X_oth, y_oth, rng, k=5, n_new=N_SYNTH_J):
    """Untuk tiap titik sintetis: apakah ia 'menyusup', dan ke kelas mana ia paling dekat."""
    k = min(k, len(X_min) - 1)
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    i = rng.integers(0, len(X_min), n_new)
    j = ind[i, rng.integers(1, k + 1, n_new)]
    lam = rng.random((n_new, 1))
    X_new = X_min[i] + lam * (X_min[j] - X_min[i])                     # persamaan (1)
    d_min, _ = NearestNeighbors(n_neighbors=1).fit(X_min).kneighbors(X_new)
    nn_o = NearestNeighbors(n_neighbors=1).fit(X_oth)
    d_oth, idx_o = nn_o.kneighbors(X_new)
    intruding = d_oth[:, 0] < d_min[:, 0]
    dest = y_oth[idx_o[:, 0]]
    return X_new, intruding, dest

CKJ = os.path.join(OUTDIR, "secJ_synthetic_destination.csv")
rng_j = np.random.default_rng(RANDOM_STATE)
rows_j, store_j = [], {}

for c in (TARGET_CATS if RUN_J_SYNTH else []):
    X_cat = X_tr[y_tr == c]
    pick = rng_j.choice(len(X_cat), size=min(N_CONTROL, len(X_cat)), replace=False)
    X_min = X_cat[pick]
    mask_o = y_tr != c
    X_oth, y_oth = X_tr[mask_o], y_tr[mask_o]
    X_new, intr, dest = synthetic_destination(X_min, X_oth, y_oth, rng_j)
    store_j[c] = (X_min, X_new, intr, dest)
    d_intr = pd.Series(dest[intr])
    row = dict(Category=c, n_synth=len(X_new),
               **{"synthetic intrusion (%)": 100 * intr.mean()})
    if len(d_intr):
        share = d_intr.value_counts(normalize=True) * 100
        row["to BENIGN (%)"] = float(share.get("BENIGN", 0.0))
        row["top non-BENIGN"] = str(share.drop(labels=["BENIGN"], errors="ignore").idxmax()) \
            if len(share.drop(labels=["BENIGN"], errors="ignore")) else ""
    else:
        row["to BENIGN (%)"] = np.nan; row["top non-BENIGN"] = ""
    rows_j.append(row)
    print(f"{c:13s} ι={row['synthetic intrusion (%)']:5.1f}%  "
          f"ke BENIGN={row['to BENIGN (%)']:5.1f}%  lain teratas={row['top non-BENIGN']}")

if rows_j:
    jd = pd.DataFrame(rows_j)
    jd.to_csv(CKJ, index=False)
    print("\n", jd.round(1).to_string(index=False))
    sub = jd.dropna(subset=["to BENIGN (%)"])
    if len(sub) >= 3:
        from scipy.stats import spearmanr
        rho, p = spearmanr(sub["synthetic intrusion (%)"], sub["to BENIGN (%)"])
        print(f"\nSpearman ι vs proporsi-ke-BENIGN: rho = {rho:.2f}, p = {p:.3f}  (n = {len(sub)})")
        print("Hipotesis didukung bila rho positif dan proporsi ke BENIGN tinggi "
              "pada kategori dengan ι besar.")

In [ ]:
# --- Proyeksi ruang fitur lokal: Infiltration dan satu kategori pembanding ---
if RUN_J_SYNTH and store_j:
    pairs = [c for c in ["Infiltration", "PortScan", "Botnet"] if c in store_j][:2]
    if len(pairs) >= 1:
        fig, axes = plt.subplots(1, len(pairs), figsize=(6.0 * len(pairs), 5.2))
        axes = np.atleast_1d(axes)
        for ax, c in zip(axes, pairs):
            X_min, X_new, intr, dest = store_j[c]
            mask_o = y_tr != c
            X_oth, y_oth = X_tr[mask_o], y_tr[mask_o]
            # latar: subsampel kelas lain, diwarnai BENIGN vs serangan lain
            sel = np.random.default_rng(0).choice(len(X_oth), size=min(4000, len(X_oth)), replace=False)
            Xb, yb = X_oth[sel], y_oth[sel]
            pca = PCA(n_components=2, random_state=0).fit(np.vstack([X_min, Xb]))
            Pb, Pm, Pn = pca.transform(Xb), pca.transform(X_min), pca.transform(X_new)
            is_ben = yb == "BENIGN"
            ax.scatter(Pb[is_ben, 0], Pb[is_ben, 1], s=6, c="#c9d6e4", label="BENIGN", zorder=1)
            ax.scatter(Pb[~is_ben, 0], Pb[~is_ben, 1], s=6, c="#f0c9a8",
                       label="serangan lain", zorder=2)
            ax.scatter(Pn[~intr, 0], Pn[~intr, 1], s=10, c="#2e75a3", alpha=.6,
                       label="sintetis (aman)", zorder=3)
            ax.scatter(Pn[intr, 0], Pn[intr, 1], s=16, c="#c0392b", alpha=.8,
                       label="sintetis (menyusup)", zorder=4)
            ax.scatter(Pm[:, 0], Pm[:, 1], s=46, facecolors="none", edgecolors="#1f4e79",
                       linewidths=1.4, label=f"{c} asli (n={len(X_min)})", zorder=5)
            ax.set_title(f"{c} — ι = {100*intr.mean():.1f}%", fontsize=11)
            ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
            ax.legend(fontsize=7, loc="best", framealpha=.9)
        fig.tight_layout()
        p = os.path.join(OUTDIR, "v3_fig_synthetic_projection.png")
        fig.savefig(p, dpi=200, bbox_inches="tight"); plt.close(fig)
        print("Gambar disimpan:", p)

## K. Ringkasan untuk naskah

Bagian ini merakit semua tabel yang akan masuk ke naskah versi berikutnya. Semuanya dihitung ulang dari
file checkpoint, tidak ada angka yang diketik tangan.

Yang paling penting adalah **tabel korelasi per model** di bagian akhir. Di situlah terlihat apakah
klaim utama naskah bertahan ketika baseline diperkuat. Tiga kemungkinan dan artinya:

* korelasi tetap kuat dan negatif pada ketiga model → klaim menguat, naskah tinggal diperluas;
* korelasi melemah tetapi arahnya sama → klaim perlu dibatasi pada keluarga model tertentu;
* korelasi hilang atau berbalik → klaim utama harus ditulis ulang, dan **itu pun temuan yang layak terbit**.


In [ ]:
from scipy.stats import wilcoxon, spearmanr

dg = pd.read_csv(CKG)
dg = dg[dg.seed.isin(SEEDS) & dg.category.isin(TARGET_CATS) & dg.model.isin(MODELS)]

def f1_from(tp, fp, fn):
    return 200.0 * tp / max(2 * tp + fp + fn, 1)

# --- Tabel K1: F1 per kategori x model x metode (digabung lintas fold dalam satu seed) ---
g = (dg.groupby(["seed", "category", "model", "method"])[["tp", "fp", "fn"]].sum().reset_index())
g["f1"] = [f1_from(*r) for r in g[["tp", "fp", "fn"]].values]
k1 = (g.groupby(["category", "model", "method"])["f1"]
        .agg(["mean", "std"]).reset_index())
k1["F1"] = k1["mean"].map("{:.1f}".format) + " ± " + k1["std"].fillna(0).map("{:.1f}".format)
piv1 = k1.pivot_table(index=["category", "model"], columns="method", values="F1", aggfunc="first")
piv1 = piv1.reindex(columns=[m for m in METHODS_G if m in piv1.columns])
print("=== K1. F1 kelas minoritas pada n = 29 (mean ± SD atas seed) ===")
print(piv1.to_string(), "\n")
piv1.to_csv(os.path.join(OUTDIR, "v3_k1_f1_by_model.csv"))

# --- Tabel K2: uji Wilcoxon SMOTE vs base, per kategori x model ---
rows = []
for (c, mo), sub in dg.groupby(["category", "model"]):
    b = sub[sub.method == "base"].set_index(["seed", "fold"]).sort_index()
    s = sub[sub.method == "smote"].set_index(["seed", "fold"]).sort_index()
    common = b.index.intersection(s.index)
    if len(common) < 6:
        continue
    b, s = b.loc[common], s.loc[common]
    f1b = np.array([f1_from(*r) for r in b[["tp", "fp", "fn"]].values])
    f1s = np.array([f1_from(*r) for r in s[["tp", "fp", "fn"]].values])
    d = f1s - f1b
    try:
        stat, p = wilcoxon(f1s, f1b) if np.any(d != 0) else (np.nan, 1.0)
    except Exception:
        stat, p = np.nan, np.nan
    fnb, fns = b["fn"].sum(), s["fn"].sum()
    rows.append(dict(category=c, model=mo, n_pairs=len(common),
                     base_F1=f1b.mean(), smote_F1=f1s.mean(), delta=d.mean(),
                     p=p, R_c=100.0 * (fnb - fns) / max(fnb, 1)))
k2 = pd.DataFrame(rows)
if len(k2):
    k2 = k2.sort_values(["model", "category"])
    k2.to_csv(os.path.join(OUTDIR, "v3_k2_wilcoxon_by_model.csv"), index=False)
    print("=== K2. SMOTE vs base: selisih F1, uji Wilcoxon berpasangan, dan R_c ===")
    print(k2.round(3).to_string(index=False), "\n")

# --- Tabel K3: korelasi antar kategori, per model (INI uji klaim utama) ---
shape = pd.read_csv(CKS) if os.path.exists(CKS) else pd.DataFrame()
comp = pd.read_csv(CKI) if os.path.exists(CKI) else pd.DataFrame()
if len(k2):
    print("=== K3. Apakah 'detection headroom' tetap memprediksi manfaat SMOTE? ===")
    print("    (korelasi Spearman antar kategori; n = jumlah kategori)\n")
    for mo in MODELS:
        sub = k2[k2.model == mo]
        if len(sub) < 4:
            continue
        line = [f"model {mo.upper():4s} (n = {len(sub)} kategori)"]
        r1, p1 = spearmanr(sub["base_F1"], sub["delta"])
        r2, p2 = spearmanr(sub["base_F1"], sub["R_c"])
        line.append(f"  baseline F1 vs ΔF1 : rho = {r1:+.2f} (p = {p1:.3f})")
        line.append(f"  baseline F1 vs R_c : rho = {r2:+.2f} (p = {p2:.3f})")
        if len(shape):
            sh = (shape[shape.category.isin(sub.category)]
                  .groupby("category")[["dispersion ratio", "synthetic intrusion (%)"]]
                  .mean().reset_index())
            m = sub.merge(sh, on="category", how="inner")
            if len(m) >= 4:
                r3, p3 = spearmanr(m["dispersion ratio"], m["R_c"])
                r4, p4 = spearmanr(m["synthetic intrusion (%)"], m["R_c"])
                line.append(f"  dispersi δ  vs R_c : rho = {r3:+.2f} (p = {p3:.3f})")
                line.append(f"  intrusi ι   vs R_c : rho = {r4:+.2f} (p = {p4:.3f})")
        if len(comp):
            m = sub.merge(comp.rename(columns={"Category": "category"}), on="category", how="inner")
            if len(m) >= 4:
                r5, p5 = spearmanr(m["N3 (%)"], m["R_c"])
                line.append(f"  overlap N3  vs R_c : rho = {r5:+.2f} (p = {p5:.3f})")
        print("\n".join(line), "\n")
    print("Catatan: pada n = 6 kategori, hanya |rho| ≳ 0,83 yang signifikan pada α = 0,05. "
          "Angka di atas adalah indikasi arah, bukan bukti kuat; interpretasikan bersama K2.")

# --- Tabel K4: eksperimen representasi ---
if os.path.exists(CKR):
    dr = pd.read_csv(CKR)
    if len(dr):
        dr["f1"] = [f1_from(*r) for r in dr[["tp", "fp", "fn"]].values]
        k4 = (dr.groupby(["category", "model", "k", "method"])["f1"].mean()
                .unstack("method"))
        if "base" in k4.columns and "smote" in k4.columns:
            k4["delta"] = k4["smote"] - k4["base"]
        k4.to_csv(os.path.join(OUTDIR, "v3_k4_representation.csv"))
        print("=== K4. Eksperimen representasi: F1 pada top-k fitur ===")
        print(k4.round(1).to_string(), "\n")
        print("Bandingkan kolom delta antar nilai k. Jika delta mengecil ketika k mengecil "
              "(representasi membaik), yang membatasi adalah ruang fitur, bukan jumlah sampel.")
print("\nSemua file hasil ada di:", OUTDIR)